## Module 3: Machine Learning for Classification

This module covers the fundamentals of **classification**, with a focus on predicting categorical outcomes from data. It introduces the typical classification workflow, including data preparation, feature analysis, model training, and evaluation. The main model used is **logistic regression**, along with techniques for handling categorical variables and interpreting model performance using metrics such as accuracy.

### Part 1: Data Preparation

In [1]:
# get telecom churn data
!wget 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv' -O data-week-3.csv

--2026-09-21 18:14:58--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.009s  

2026-09-21 18:14:58 (103 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [4]:
# load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# check the data
df = pd.read_csv('data-week-3.csv')
# since columns are too many and hidden, use transpose
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [5]:
# rewrite column names lowercase with no space (to be able to use in dot form)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# check categorical columns and make their content lowercase with no space
cat_cols = list(df.dtypes[df.dtypes == 'str'].index)
for col in cat_cols:
    df[col] = df[col].str.lower().str.replace(' ', '_')

df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [ ]:
# check data types to clean the data 
# (totalcharges should be numeric but str, churn should be 1 or 0)
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [25]:
# turn column into numeric and ignore errors ('_' ones), fill missing with 0 (although not the best option)
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce').fillna(0)

# turn column into numeric rather than yes or no
df.churn = (df.churn == 'yes').astype(int)

### Part 2: Setting Up the Validation Framework

In [ ]:
from sklearn.model_selection import train_test_split

# split data with sckit-learn (since sckitlearn func divides 2, do 2 times)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1) # 1/5
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1) # 1/4 of remaining 4/5

# reset index for each
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# separate target variable
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

# delete target variable data from features
del df_train['churn']
del df_val['churn']
del df_test['churn']

# get the size of each set
print("Total size:", len(df))
print(f"Train, Val, Test: ({len(df_train)}, {len(df_val)}, {len(df_test)})")
print(f"y_train, y_val, y_test: ({len(y_train)}, {len(y_val)}, {len(y_test)})")

Total size: 7043
Train, Val, Test: (4225, 1409, 1409)
y_train, y_val, y_test: (4225, 1409, 1409)


### Part 3: Exploratory Data Analysis (EDA)

In [27]:
# use data except test one, check for null values
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [28]:
# check mean and distribution of target variable
print('Mean churn (churn rate of customers):', df_full_train.churn.mean())
df_full_train.churn.value_counts(normalize=True)

Mean churn (churn rate of customers): 0.26996805111821087


churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [ ]:
# separate numerical and categorical columns
numerical = ['tenure', 'monthlycharges', 'totalcharges']
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

# check number of unique values for all categorical columns
df_full_train[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

### Part 4: Feature Importance - Churn Rate and Risk Ratio
#### Churn Rate:

In [ ]:
# check columns again
df_full_train.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
1,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
2,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
3,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
4,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [42]:
churn_female = df_full_train[df_full_train.gender == 'female'].churn.mean()
churn_male = df_full_train[df_full_train.gender == 'male'].churn.mean()
global_churn = df_full_train.churn.mean()

print(f"Female churn rate: {churn_female * 100:.2f}%")
print(f"Male churn rate: {churn_male * 100:.2f}%")
print(f"Global churn rate: {global_churn * 100:.2f}%")
print(f"Global - Female: {(global_churn - churn_female) * 100:.2f}%")
print(f"Global - Male: {(global_churn - churn_male) * 100:.2f}%")

Female churn rate: 27.68%
Male churn rate: 26.32%
Global churn rate: 27.00%
Global - Female: -0.69%
Global - Male: 0.68%


In [43]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

print(f"With partner rate: {churn_partner * 100:.2f}%")
print(f"With NO partner rate: {churn_no_partner * 100:.2f}%")
print(f"Global - with partner: {(global_churn - churn_partner) * 100:.2f}%")
print(f"Global - with NO partner: {(global_churn - churn_no_partner) * 100:.2f}%")

df_full_train.partner.value_counts()

With partner rate: 20.50%
With NO partner rate: 32.98%
Global - with partner: 6.49%
Global - with NO partner: -5.98%


partner
no     2932
yes    2702
Name: count, dtype: int64

Compare each group’s churn rate with the global churn rate:

* `diff > 0` / `diff < 0` → tells whether the group churns above or below average
* `|diff|` → tells how far the group is from the global rate
* Larger `|diff|` → stronger relationship with churn

Gender has very small absolute differences (~0.007), while partner status has much larger ones (~0.06). So partner status appears more informative for churn than gender.

#### Risk Ratio:

Risk ratio compares a group’s churn rate with the global churn rate:

* `risk > 1` → group churns more than average
* `risk < 1` → group churns less than average
* Farther from `1` → stronger relationship with churn

A risk ratio close to `1` means the group behaves similarly to the overall population.


In [45]:
print(f"churn_no_partner / global_churn: {churn_no_partner / global_churn}")
print(f"churn_partner / global_churn: {churn_partner / global_churn}")

churn_no_partner / global_churn: 1.2216593879412643
churn_partner / global_churn: 0.7594724924338315


```sql
SELECT
    gender,
    AVG(churn),
    AVG(churn) - global_churn AS diff,
    AVG(churn) / global_churn AS risk
FROM
    data
GROUP BY
    gender;
```

If does not show output below, use `from IPython.display import display`

In [46]:
# get above sql in pd
for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


### Part 5: Feature Importance - Mutual Information

It is the concept from information theory, which tells us how much we can learn about one variable if we know the value of another.

https://en.wikipedia.org/wiki/Mutual_information

In [ ]:
from sklearn.metrics import mutual_info_score

mutual_info_score(df_full_train.churn, df_full_train.contract)
mutual_info_score(df_full_train.gender, df_full_train.churn)
mutual_info_score(df_full_train.contract, df_full_train.churn)
mutual_info_score(df_full_train.partner, df_full_train.churn)

In [ ]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

In [ ]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

### Part 6: Feature Importance - Correlation

In [ ]:
df_full_train.tenure.max()
df_full_train[numerical].corrwith(df_full_train.churn).abs()
df_full_train[df_full_train.tenure <= 2].churn.mean()
df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()
df_full_train[df_full_train.tenure > 12].churn.mean()
df_full_train[df_full_train.monthlycharges <= 20].churn.mean()
df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean()
df_full_train[df_full_train.monthlycharges > 50].churn.mean()

### Part 7: One-hot Encoding

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

### Part 8: Logistic Regression

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-7, 7, 51)
sigmoid(10000)
plt.plot(z, sigmoid(z))

In [ ]:
def linear_regression(xi):
    result = w0
    
    for j in range(len(w)):
        result = result + xi[j] * w[j]
        
    return result

In [ ]:
def logistic_regression(xi):
    score = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result

### Part 9: Training Logistic Regression with Scikit-Learn

### Part 10: Model Interpretation

### Part 11: Using the Model